In [ ]:
import os
import getpass
import subprocess
import sys

try:
    from dotenv import load_dotenv
except ModuleNotFoundError:
    print("python-dotenv is not installed. Installing it now...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "python-dotenv"])
    from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY not found in .env file. Please create a .env file with your API key.")

os.environ["GROQ_API_KEY"] = api_key

assert os.getenv("GROQ_API_KEY"), "Groq API key is missing."
print("Groq API key is configured from .env file.")

Groq API key is configured for this notebook session.


In [2]:
from typing import TypedDict
from pathlib import Path

from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END

GROQ_MODEL = "openai/gpt-oss-120b"

groq_model = ChatGroq(
    model=GROQ_MODEL,
    temperature=0.2,
    max_tokens=1800,
    reasoning_format="parsed",
    max_retries=2,
)

class QAAgentState(TypedDict):
    requirement: str
    analysis: str
    test_cases: str
    security_review: str
    review: str


def call_specialist(system_prompt, task):
    response = groq_model.invoke([
        ("system", system_prompt),
        ("human", task),
    ])
    return response.content


def requirements_analyst(state: QAAgentState):
    analysis = call_specialist(
        "You are a senior QA requirements analyst. Identify actors, business rules, acceptance criteria, risks, dependencies, and ambiguous requirements. Be concise and do not invent missing facts.",
        f"Analyze this requirement for testing:\n\n{state['requirement']}",
    )
    return {"analysis": analysis}


def test_designer(state: QAAgentState):
    test_cases = call_specialist(
        "You are a senior test designer. Produce a compact Markdown table with ID, scenario, preconditions, steps, expected result, test type, and priority. Cover positive, negative, boundary, security, and failure paths.",
        f"Requirement:\n{state['requirement']}\n\nRequirements analysis:\n{state['analysis']}\n\nDesign executable test cases.",
    )
    return {"test_cases": test_cases}


def security_reviewer(state: QAAgentState):
    security_review = call_specialist(
        "You are a security reviewer for software requirements. Identify security, privacy, access control, session handling, and data exposure risks. Highlight missing safeguards and note any conflicting or vague requirements.",
        f"Requirement:\n{state['requirement']}\n\nRequirements analysis:\n{state['analysis']}\n\nProposed tests:\n{state['test_cases']}",
    )
    return {"security_review": security_review}


def qa_reviewer(state: QAAgentState):
    review = call_specialist(
        "You are a critical QA lead. Review the proposed tests for requirement coverage, missing edge cases, duplication, testability, and business risk. Finish with APPROVE or REVISE and a short reason.",
        f"Requirement:\n{state['requirement']}\n\nAnalysis:\n{state['analysis']}\n\nProposed tests:\n{state['test_cases']}\n\nSecurity review:\n{state['security_review']}",
    )
    return {"review": review}


builder = StateGraph(QAAgentState)
builder.add_node("requirements_analyst", requirements_analyst)
builder.add_node("test_designer", test_designer)
builder.add_node("security_reviewer", security_reviewer)
builder.add_node("qa_reviewer", qa_reviewer)
builder.add_edge(START, "requirements_analyst")
builder.add_edge("requirements_analyst", "test_designer")
builder.add_edge("test_designer", "security_reviewer")
builder.add_edge("security_reviewer", "qa_reviewer")
builder.add_edge("qa_reviewer", END)

qa_agent_chain = builder.compile()
print(f"Four-agent QA chain is ready with {GROQ_MODEL}.")


Four-agent QA chain is ready with openai/gpt-oss-120b.


In [4]:
requirements_path = Path("requirements_document.md")
if not requirements_path.exists():
    requirements_path = Path("requirements_document.txt")

if not requirements_path.exists():
    raise FileNotFoundError("Could not find a requirements document. Create requirements_document.md or requirements_document.txt in the workspace.")

requirements_text = requirements_path.read_text(encoding="utf-8")

result = qa_agent_chain.invoke({
    "requirement": requirements_text,
    "analysis": "",
    "test_cases": "",
    "security_review": "",
    "review": "",
})

output_file = Path("qa_agent_output.txt")

with open(output_file, "w", encoding="utf-8") as f:
    for heading, key in [
        ("REQUIREMENTS ANALYST", "analysis"),
        ("TEST DESIGNER", "test_cases"),
        ("SECURITY REVIEWER", "security_review"),
        ("QA REVIEWER", "review"),
    ]:
        separator = f"\n{'=' * 20} {heading} {'=' * 20}\n"
        f.write(separator)
        f.write(result[key])
        f.write("\n")
        
        print(separator)
        print(result[key])

print(f"\n✓ Output also saved to qa_agent_output.txt")


==================== REQUIREMENTS ANALYST ====================

**Actors**
| # | Actor | Role / Interaction |
|---|-------|--------------------|
| 1 | **Registered Customer** | Initiates password‑reset request, clicks the email link, enters a new password. |
| 2 | **Password‑Reset Service** (backend) | Generates secure token, validates link, enforces single‑use & expiration, updates password. |
| 3 | **Email Delivery System** | Sends the reset email containing the token‑link. |
| 4 | **Account‑Lock Service** (optional) | Determines if an account is temporarily locked. |
| 5 | **Business Team / Product Owner** | Provides final password‑policy and clarifies lock‑handling rules. |

---

### Business Rules (as stated / inferred)

| # | Rule | Source |
|---|------|--------|
| B1 | Reset link must be **time‑limited**. | “expire after 15 minutes” |
| B2 | Reset link must be **single‑use** – cannot be reused after a successful reset. | “must not work after it has been used once” |
| B3 | Syst